# Balanced Mixed cho TBPS

Notebook chạy các pipeline IRRA, DM-Adapter, RDE và ITSELF trên RSTPReid theo đúng thứ tự này. Mỗi phương pháp giữ cấu hình train riêng và dùng chung chính sách Balanced Mixed. Cell IRRA đang mở; ba cell còn lại được comment. Bỏ comment cell của phương pháp bạn muốn chạy.

Mỗi cell là một lượt train riêng 60 epochs, có thể vượt thời lượng một Kaggle session. Mining dùng trọng số model hiện tại và cập nhật mỗi ba epochs.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/clip-balanced-mixed")
if not repo.exists():
    subprocess.run([
        "git", "clone", "https://github.com/lqb464/clip-balanced-mixed.git", str(repo)
    ], check=True)

requirements = repo / "requirements.txt"
dependencies = []
for line in requirements.read_text(encoding="utf-8").splitlines():
    package = line.strip()
    if package and not package.startswith(("#", "torch")):
        dependencies.append(package)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *dependencies], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle trước khi chạy cell huấn luyện.")

data_root = Path("/kaggle/input/datasets/hoanggv/tbps-benchmark/benchmark")
if not (data_root / "RSTPReid" / "data_captions.json").is_file():
    raise FileNotFoundError(f"Không tìm thấy annotation RSTPReid tại {data_root}")

os.environ["TBPS_REPO"] = str(repo)
os.environ["TBPS_DATA_ROOT"] = str(data_root)
print(f"GPU đang dùng: {torch.cuda.get_device_name(0)}")
print(f"Repo: {repo}")
print(f"Dataset: {data_root / 'RSTPReid'}")


## 1. IRRA

Batch size 64; `sdm+mlm+id`; bật image augmentation và MLM.


In [ ]:
%%bash
set -euo pipefail
cd "$TBPS_REPO/irra"
CUDA_VISIBLE_DEVICES=0 python -u train.py \
    --name balanced_mixed_rstpreid \
    --img_aug \
    --batch_size 64 \
    --MLM \
    --dataset_name RSTPReid \
    --loss_names 'sdm+mlm+id' \
    --num_epoch 60 \
    --root_dir "$TBPS_DATA_ROOT" \
    --sampler balanced_mixed \
    --positive-pairs 4 \
    --sampler-refresh-every 3 \
    --seed 1


## 2. DM-Adapter

Batch size 128; `sdm+aux`; bật image augmentation và MLM.


In [ ]:
# %%bash
# set -euo pipefail
# cd "$TBPS_REPO/dm-adapter"
# CUDA_VISIBLE_DEVICES=0 python -u train.py \
#     --name balanced_mixed_rstpreid \
#     --img_aug \
#     --batch_size 128 \
#     --MLM \
#     --dataset_name RSTPReid \
#     --loss_names 'sdm+aux' \
#     --num_epoch 60 \
#     --root_dir "$TBPS_DATA_ROOT" \
#     --lr 3e-4 \
#     --num_experts 6 \
#     --topk 2 \
#     --reduction 8 \
#     --sampler balanced_mixed \
#     --positive-pairs 4 \
#     --sampler-refresh-every 3 \
#     --seed 1


## 3. RDE

Batch size 64; cấu hình TAL/RBS từ launcher RDE trong repo; noise rate 0.


In [ ]:
# %%bash
# set -euo pipefail
# cd "$TBPS_REPO/rde/2024-CVPR-RDE"
# CUDA_VISIBLE_DEVICES=0 python -u train.py \
#     --noisy_rate 0.0 \
#     --noisy_file ./noiseindex/RSTPReid_0.0.npy \
#     --name RDE-Balanced-Mixed \
#     --img_aug \
#     --txt_aug \
#     --batch_size 64 \
#     --select_ratio 0.3 \
#     --tau 0.015 \
#     --root_dir "$TBPS_DATA_ROOT" \
#     --output_dir run_logs_balanced_mixed \
#     --margin 0.1 \
#     --dataset_name RSTPReid \
#     --loss_names 'TAL+sr0.3_tau0.015_margin0.1_n0.0' \
#     --num_epoch 60 \
#     --lr-total-epochs 60 \
#     --sampler balanced_mixed \
#     --positive-pairs 4 \
#     --sampler-refresh-every 3 \
#     --seed 1


## 4. ITSELF

Batch size 256; `tal+cid`; bật `only_global`.


In [ ]:
# %%bash
# set -euo pipefail
# cd "$TBPS_REPO/itself"
# CUDA_VISIBLE_DEVICES=0 python -u train.py \
#     --name ITSELF-Balanced-Mixed \
#     --output_dir ITSELF-balanced-mixed \
#     --batch_size 256 \
#     --dataset_name RSTPReid \
#     --root_dir "$TBPS_DATA_ROOT" \
#     --loss_names 'tal+cid' \
#     --num_epoch 60 \
#     --only_global \
#     --sampler balanced_mixed \
#     --positive-pairs 4 \
#     --sampler-refresh-every 3 \
#     --seed 1
